Exemplo de coordenação entre múltiplos agentes de IA utilizando a CrewAI

In [49]:
!pip install -U crewai ddgs openai python-dotenv


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.1 MB/s eta 0:00:00


In [50]:
import os
import base64
from dotenv import load_dotenv
from openai import OpenAI
from ddgs import DDGS
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from langchain_openai import ChatOpenAI
import uuid

Configuração

In [51]:
os.environ["OPENAI_API_KEY"]="SUA-CHAVE-AQUI"

LLM

In [52]:
llm = LLM(
    model="gpt-3.5-turbo",
    temperature=0
)
openai_client = OpenAI()

Tools que serão usadas pelos Agents

In [53]:
# ============================================================ # TOOL DE PESQUISA # ============================================================

@tool("Pesquisa no DuckDuckGo")
def duckduckgo_search(query: str) -> str:
  """
    Pesquisa informações na Internet usando o DuckDuckGo.
    Utilize esta ferramenta quando precisar encontrar:
      - informações atuais;
      - notícias;
      - tecnologias;
      - empresas; - artigos;
      - fontes na Internet.
    """
  try:
    results = DDGS().text( query, max_results=8 )
    if not results: return "Nenhum resultado encontrado."

    output = []
    for index, result in enumerate( results, start=1 ):
      title = result.get( "title", "" )
      body = result.get( "body", "" )
      href = result.get( "href", "" )
      output.append(
          f"""
          RESULTADO {index}
          Título: {title}
          Resumo: {body}
          URL: {href} """
      )

    return "\n".join(output)
  except Exception as e:
    return ( f"Erro ao pesquisar no DuckDuckGo: {e}" )


In [54]:
# ============================================================ # TOOL DE GERAÇÃO DE IMAGEM # ============================================================ #

@tool("Geração de Imagem")
def generate_image(prompt: str) -> str:
  """
    Gera uma imagem utilizando a API de imagens da OpenAI.
    Recebe uma descrição textual da imagem e salva o resultado como um arquivo PNG.
  """
  try:
    response = openai_client.images.generate( model="gpt-image-1", prompt=prompt, size="1024x1024" )
    image_data = response.data[0]
    # ---------------------------------------------------- # A API retornou Base64 # ----------------------------------------------------
    if getattr( image_data, "b64_json", None ):
      image_bytes = base64.b64decode( image_data.b64_json )
    # Gera um identificador único
    image_id = uuid.uuid4().hex
    filename = f"imagem_gerada_{image_id}.png"
    with open( filename, "wb" ) as file: file.write( image_bytes )
    return ( f"Imagem criada com sucesso.\n" f"Arquivo: {filename}" )
    # ---------------------------------------------------- # A API retornou URL # ----------------------------------------------------
    if getattr( image_data, "url", None ):
      return ( "Imagem criada com sucesso.\n" f"URL: {image_data.url}" )

    return ( "A imagem foi gerada, mas não foi " "possível obter o arquivo ou URL." )
  except Exception as e:
    return ( f"Erro ao gerar imagem: {e}" )


Agents

In [55]:
roteirista_agent = Agent(
    role="Roteirista",
    llm=llm,
    tools=[duckduckgo_search],
    goal="""
     Pesquisar e elaborar um roteiro detalhado para um vídeo completo no YouTube.
    """,
    backstory="""
      Especialista em criação de conteúdo e storytelling para vídeos de games.
    """,
    verbose=True
)

In [56]:
designer_agent = Agent(
    role="Thumbnail",
    llm=llm,
    tools=[generate_image],
    goal="""
      Utilizar o roteiro gerado para produzir três opções de thumbnails.
      As thumbnails devem ser inspiradas no conteúdo do vídeo, destacando elementos visuais atrativos.
  """,
  backstory="""
    Designer gráfico com experiência em thumbnails chamativas para YouTube.
  """,
  verbose=True
)

In [57]:
revisor_agent = Agent(
    role="Revisor",
    llm=llm,
    goal="""
      Revisar o texto do roteiro
      Escrever a versão contendo roteiro + thumbnails
  """,
  backstory="""
    Revisor de criação de conteúdo e storytelling para vídeos de games.
  """,
  verbose=True
)

Tasks

In [58]:
roteirista_task = Task(
    description="""
      Elaborar um roteiro detalhado para um vídeo completo no YouTube para tema
      {tema}.
    """,

    expected_output="""
      Roteiro detalhado para um vídeo completo no YouTube para o tema informado.
    """,

    agent=roteirista_agent,
    verbose=True
)

In [59]:
designer_task = Task(
    description="""
      Utilizar o roteiro gerado para produzir três opções de thumbnails do tema {tema}.
      As thumbnails devem ser inspiradas no conteúdo do vídeo, destacando elementos visuais atrativos.
    """,

    expected_output="""
      Três opções de thumbnails do tema {tema}.
    """,
    agent=designer_agent,
    context=[roteirista_task],
    verbose=True
)

In [60]:
revisor_task = Task(
    description="""
      Revisar o texto do roteiro
      Escrever a versão contendo roteiro + thumbnails
    """,

    expected_output="""
      Roteiro revisado e uma thumbnail escolhida entre as três geradas pelo designer_task.
    """,
    agent=revisor_agent,
    context=[roteirista_task, designer_task],
    verbose=True
)

Coordenação de Crew

In [61]:
crew = Crew(
    name="Media Makers",
    agents=[roteirista_agent, designer_agent, revisor_agent],
    tasks=[roteirista_task, designer_task, revisor_task],
    process=Process.sequential,
    verbose=True,
    max_iter=15
)

In [63]:
result = await crew.kickoff_async(inputs={ "tema": """ 'Melhores jogos de 2025 """ })

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: Media Makers                                                                                             │
│  ID: 7d72e38c-58eb-47ca-b0a8-fd8b7c9c75c4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│        Elaborar um roteiro detalhado para um vídeo completo no YouTube para tema                                │
│         'Melhores jogos de 2025 .                                                                               │
│                                                                                                                 │
│  ID: 0b223edb-4339-4099-b63b-fab26412be05                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Roteirista                                                                                              │
│                                                                                                                 │
│  Task:                                                                                                          │
│        Elaborar um roteiro detalhado para um vídeo completo no YouTube para tema                                │
│         'Melhores jogos de 2025 .                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: pesquisa_no_duck_duck_go                                                                                 │
│  Args: {'query': 'Melhores jogos de 2025'}                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool pesquisa_no_duck_duck_go executed with result:  
          RESULTADO 1 
          Título: Os 25 melhores jogos de 2025 - TheGamer Br 
          Resumo: Dec 27, 2025 · Confira os melhores jogos de 2025: destaques como Clair Obscur, Death Stranding ...


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Revisor                                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│        Revisar o texto do roteiro                                                                               │
│        Escrever a versão contendo roteiro + thumbnails                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Revisor                                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Roteiro para Vídeo no YouTube: Melhores Jogos de 2025                                                        │
│                                                                                                                 │
│  ## Introdução                                                                                                  │
│  - Abertura com música animada e visual dinâmico.                                                               │
│  - Apresentação do tema: "Os Melhores Jogos de 2025".                                                           │
│  - Breve contextualização sobre a importância dos jogos na cultura atual.                                       │
│                                                                                                                 │
│  ## Lista dos Melhores Jogos de 2025                                                                            │
│  1. **Clair Obscur**                                                                                            │
│     - Apresentação do jogo e seus destaques.                                                                    │
│     - Comentários sobre a jogabilidade e enredo.                                                                │
│     - Imagens e vídeos do gameplay.                                                                             │
│     - Thumbnail para o jogo Clair Obscur, destacando elementos misteriosos e futuristas.                        │
│     ![Clair Obscur Thumbnail](sandbox:/imagem_gerada_32cbf47b9c684758a5d1a4e5dd185c6e.png)                      │
│                                                                                                                 │
│  2. **Death Stranding 2**                                                                                       │
│     - Breve histórico do primeiro jogo.                                                                         │
│     - Novidades e melhorias no segundo título.                                                                  │
│     - Opiniões da crítica especializada.                                                                        │
│     - Thumbnail para o jogo Death Stranding 2, com visual impactante e elementos de ficção científica.          │
│     ![Death Stranding 2 Thumbnail](sandbox:/imagem_gerada_40a3bc6397cb45379dd7a0858f094b79.png)                 │
│                                                                                                                 │
│  3. **Hades 2**                                                                                                 │
│     - Exploração da mitologia presente no jogo.                                                                 │
│     - Análise da mecânica de combate.                                                                           │
│     - Comparação com o jogo original.                                                                           │
│     - Thumbnail para o jogo Hades 2, com foco na mitologia e elementos de combate emocionantes.                 │
│     ![Hades 2 Thumbnail](sandbox:/imagem_gerada_292bbe46dc5749faa6565f8b507c34e3.png)                           │
│                                                                                                                 │
│  4. **Hollow Knight Silksong**                         

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│        Revisar o texto do roteiro                                                                               │
│        Escrever a versão contendo roteiro + thumbnails                                                          │
│                                                                                                                 │
│  Agent: Revisor                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: Media Makers                                                                                             │
│  ID: 7d72e38c-58eb-47ca-b0a8-fd8b7c9c75c4                                                                       │
│  Final Output: # Roteiro para Vídeo no YouTube: Melhores Jogos de 2025                                          │
│                                                                                                                 │
│  ## Introdução                                                                                                  │
│  - Abertura com música animada e visual dinâmico.                                                               │
│  - Apresentação do tema: "Os Melhores Jogos de 2025".                                                           │
│  - Breve contextualização sobre a importância dos jogos na cultura atual.                                       │
│                                                                                                                 │
│  ## Lista dos Melhores Jogos de 2025                                                                            │
│  1. **Clair Obscur**                                                                                            │
│     - Apresentação do jogo e seus destaques.                                                                    │
│     - Comentários sobre a jogabilidade e enredo.                                                                │
│     - Imagens e vídeos do gameplay.                                                                             │
│     - Thumbnail para o jogo Clair Obscur, destacando elementos misteriosos e futuristas.                        │
│     ![Clair Obscur Thumbnail](sandbox:/imagem_gerada_32cbf47b9c684758a5d1a4e5dd185c6e.png)                      │
│                                                                                                                 │
│  2. **Death Stranding 2**                                                                                       │
│     - Breve histórico do primeiro jogo.                                                                         │
│     - Novidades e melhorias no segundo título.                                                                  │
│     - Opiniões da crítica especializada.                                                                        │
│     - Thumbnail para o jogo Death Stranding 2, com visual impactante e elementos de ficção científica.          │
│     ![Death Stranding 2 Thumbnail](sandbox:/imagem_gerada_40a3bc6397cb45379dd7a0858f094b79.png)                 │
│                                                                                                                 │
│  3. **Hades 2**                                                                                                 │
│     - Exploração da mitologia presente no jogo.                                                                 │
│     - Análise da mecânica de combate.                                                                           │
│     - Comparação com o jogo original.                                                                           │
│     - Thumbnail para o jogo Hades 2, com foco na mitologia e elementos de combate emocionantes.                 │
│     ![Hades 2 Thumbnail](sandbox:/imagem_gerada_292bbe46dc5749faa6565f8b507c34e3.png)                           │
│                                                                                                                 │
│  4. **Hollow Knight Silksong**                        